# TP 04 | Pourquoi ce lot a-t-il été signalé ? (corrigé)

**Chapitre 3. Comparer des modèles prédictifs**

Chez Nervure Composites, Camille doit pouvoir expliquer une priorité d’inspection au chef de presse. Elle souhaite aussi savoir ce que l’atelier gagnerait à combiner plusieurs arbres. Vous lui remettrez le parcours d’un lot et une présélection de deux modèles avec leur seuil, leur charge et leur coût.

**Pour démarrer.** Ouvrez le notebook dans VS Code, sélectionnez le noyau Python de `.venv` et exécutez les cellules dans l’ordre. Vous aurez besoin de savoir lire un tableau et compter les faux positifs et faux négatifs. Les CSV du dépôt suffisent, même si vous ouvrez ce TP directement. Le chemin essentiel passe par le calcul de Gini, la lecture d’une feuille, la comparaison arbre-forêt-boosting, puis la note de présélection. Les quatre défis de fin peuvent se traiter dans l’ordre de votre choix.

## 1. Comment l’arbre choisit-il une question ?

**Atelier 4.1 du support MLB.** Sections 1 à 2.

![Deux coupures candidates sur les mêmes quatre lots](../figures/tp04-coupures.svg)

Une **coupure** divise un groupe en deux par une question binaire, par exemple « température ≤ 71 °C ? ». Au départ tous les lots sont dans la **racine**. Chaque groupe issu d’une coupure est un **nœud enfant**. Un groupe où le modèle s’arrête est une **feuille**. Dans le dessin, les symboles donnent les verdicts réels de quatre lots fictifs.

L’**impureté de Gini** mesure le mélange des classes dans un nœud. Si la proportion de défauts vaut $p$, elle s’écrit

$$
G(p)=1-p^2-(1-p)^2.
$$

Elle vaut zéro pour une seule classe et atteint 0,5 pour deux classes à égalité. Pour comparer deux questions, on retire du Gini du parent la moyenne des Gini des enfants **pondérée par leurs effectifs** :

$$
\Delta G=G_{parent}-\frac{n_g G_g+n_d G_d}{n_g+n_d}.
$$

À chaque nœud, l’arbre cherche un gain élevé parmi les coupures autorisées. Il fait ce choix localement, sans explorer tous les arbres possibles.

### À vous | Faire la première coupure à la main

Quatre températures valent 66, 69, 73 et 76 °C. Les deux premiers lots sont conformes et les deux derniers défectueux. Pour les seuils 67 et 71, écrivez les verdicts envoyés de chaque côté. Calculez le gain puis vérifiez-le en Python. Pourquoi donner le même poids aux deux enfants serait-il une erreur à 67 °C ?

In [ ]:
# Comparer deux questions sur un exemple dont on peut recompter chaque verdict.
import numpy as np

temperatures = np.array([66, 69, 73, 76])
verdicts = np.array([0, 0, 1, 1])


def gini(groupe):
    p = groupe.mean()
    return 2 * p * (1 - p)


# Pour chaque seuil, construisez le masque de gauche puis son complément.
for seuil in [67, 71]:
    gauche = temperatures <= seuil
    groupes = [verdicts[gauche], verdicts[~gauche]]
    apres = sum(len(g) * gini(g) for g in groupes) / len(verdicts)
    print(
        seuil, [g.tolist() for g in groupes], "gain :", round(gini(verdicts) - apres, 3)
    )

**Corrigé.** À 67 °C, on obtient [0] et [0, 1, 1]. Le second enfant contient trois lots sur quatre. Son impureté de 4/9 compte donc pour 3/4, soit une impureté finale de 1/3. Le gain vaut 1/6. À 71 °C, les enfants [0, 0] et [1, 1] sont purs : le gain atteint 0,5. Cette séparation parfaite porte sur quatre exemples fictifs. Il reste à tester les coupures apprises sur de vrais lots du fichier.

### Enquête | Parmi quelles coupures l’arbre choisit-il ?

![Gain de Gini obtenu aux trois milieux entre températures successives](../figures/tp04-recherche.svg)

Deux seuils différents peuvent envoyer exactement les mêmes lots à gauche. Sur un capteur continu, il suffit donc d’essayer un représentant par intervalle entre valeurs distinctes. Sur nos quatre panneaux, scikit-learn peut départager les partitions en évaluant leurs gains.

1. Rangez les températures et calculez les trois milieux entre voisins. Pourquoi 70 °C et 71 °C produisent-ils ici la même partition ?
2. Pour chaque milieu, retrouvez les effectifs et le gain pondéré. Lisez d’abord le maximum sur le graphique, puis vérifiez par le calcul.
3. Imaginez que le laboratoire rectifie le dernier verdict de 1 à 0. Recalculez les trois gains. Le meilleur emplacement reste-t-il unique ?

In [ ]:
# Explorer toutes les partitions distinctes des quatre lots fictifs.
milieux = (temperatures[:-1] + temperatures[1:]) / 2


def gain_coupure(y, t):
    gauche = temperatures <= t
    return gini(y) - (
        gauche.mean() * gini(y[gauche]) + (~gauche).mean() * gini(y[~gauche])
    )


# Parcourez milieux avec les verdicts initiaux puis avec [0, 0, 1, 0].
for labels in (verdicts, np.array([0, 0, 1, 0])):
    print([(t, round(gain_coupure(labels, t), 3)) for t in milieux])

**Corrigé.** Les seuils 67,5, 71 et 74,5 donnent des gains de 0,167, 0,500 et 0,167. Entre 69 et 73, aucune observation ne change de branche. Après rectification, les gains valent environ 0,042, 0,125 et 0,042. Le centre reste le meilleur choix parmi les trois mais le gain a fortement diminué. Une branche regroupe maintenant deux verdicts différents.

## 2. Suivre un lot dans un arbre entraîné

![Arbre de profondeur deux ajusté sur les 720 lots Nervure](../figures/tp04-arbre.svg)

Lisez l’arbre de haut en bas. À chaque question, partez à gauche si la condition est vraie, à droite sinon. Chaque feuille rassemble des lots anciens ayant suivi le même chemin. Le **score de défaut** de cet arbre non pondéré est la fraction de défauts parmi ces lots. Deux lots de la même feuille reçoivent donc le même score.

Reprenons les 720 lots d’entraînement et les 240 suivants en validation. Le verdict arrive huit heures après la production. Il sert de cible. Les entrées sont les trois mesures et la presse. On remplace les mesures absentes par les médianes du train. On représente la presse B par 1 et la presse A par 0. On conserve les unités des capteurs : l’arbre compare chaque mesure à un seuil dans sa propre unité.

In [ ]:
# Préparer les mêmes entrées pour tous les modèles, à partir des seules données anciennes.
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

racine = Path.cwd()
if racine.name == "corrections":
    racine = racine.parent


def lire(nom):
    return pd.read_csv(
        racine / "donnees" / nom, parse_dates=["date", "label_disponible"]
    )


train = lire("lots_train.csv")
val = lire("lots_validation.csv")
assert train.label_disponible.max() < val.date.min()
mesures = ["temperature", "pression", "duree"]
imputation = SimpleImputer(strategy="median").fit(train[mesures])


def entrees(lots):
    assert lots.ligne.isin(["A", "B"]).all()
    x = pd.DataFrame(
        imputation.transform(lots[mesures]), columns=mesures, index=lots.index
    )
    return x.assign(presse_B=lots.ligne.eq("B").astype(int))


x_train, x_val = entrees(train), entrees(val)


def ajuster_arbre(profondeur):
    return DecisionTreeClassifier(
        max_depth=profondeur, min_samples_leaf=12, random_state=42
    ).fit(x_train, train.defaut)


arbre = ajuster_arbre(2)
print(len(train), len(val), arbre.get_n_leaves())

In [ ]:
# Produire une trace lisible en suivant exactement les comparaisons de l’arbre.
def expliquer(modele, ligne):
    t = modele.tree_
    noeud = 0
    trace = []
    while t.children_left[noeud] != t.children_right[noeud]:
        variable = x_train.columns[t.feature[noeud]]
        seuil = t.threshold[noeud]
        valeur = ligne[variable]
        gauche = valeur <= seuil
        trace.append((variable, valeur, seuil, "gauche" if gauche else "droite"))
        noeud = t.children_left[noeud] if gauche else t.children_right[noeud]
    return trace, noeud


# Choisir le premier lot du fichier qui atteint la feuille au score le plus élevé.
scores_arbre = arbre.predict_proba(x_val)[:, 1]
position = int(np.argmax(scores_arbre))
trace, feuille = expliquer(arbre, x_val.iloc[position])
display(val.iloc[[position]][["id", *mesures, "ligne"]])
display(pd.DataFrame(trace, columns=["mesure", "valeur", "seuil", "branche"]).round(3))

### À vous | Justifier l’alerte avec les données de la feuille

Suivez ce lot sur le dessin avant de lire la trace. Utilisez ensuite `arbre.apply(x_train)` pour retrouver les lots anciens arrivés dans la même feuille. Recomptez les défauts et calculez le score. La règle provisoire « inspecter si le score atteint 0,50 » déclenche-t-elle une alerte ? Peut-on affirmer que ce panneau est défectueux ?

In [ ]:
# Reconstituer le score de la feuille à partir des verdicts anciens.
feuilles_train = arbre.apply(x_train)
# Sélectionnez train.loc[feuilles_train == feuille, "defaut"] et calculez size, sum, mean.
historique = train.loc[feuilles_train == feuille, "defaut"]
print(
    val.iloc[position].id, feuille, historique.size, historique.sum(), historique.mean()
)
assert np.isclose(historique.mean(), scores_arbre[position])

**Corrigé.** Le lot L0721 aboutit à la feuille 6. Elle contient 102 défauts sur 136 lots anciens, soit un score de 75.0%. L’alerte au seuil de 0,50 est justifiée par cette fréquence. Le verdict individuel viendra du laboratoire. Les seuils de la trace sont arrondis pour la lecture ; le modèle compare les valeurs en pleine précision.

### Expérience | Une petite variation du capteur change-t-elle l’alerte ?

![Score en escalier obtenu en variant une seule mesure du panneau étudié](../figures/tp04-marches.svg)

Gardez les autres capteurs et la presse du lot étudié fixes. Faites varier uniquement la mesure utilisée à la racine. On obtient une **analyse de sensibilité locale** : elle décrit la réaction du modèle à des entrées simulées. Elle ne permet pas d’affirmer qu’un réglage de la presse causerait un défaut.

Lisez sur la courbe un intervalle où le score reste constant. Créez ensuite deux copies du lot, l’une juste avant et l’autre juste après la coupure de la racine. Relevez la feuille, le score et la décision au seuil de 0,50. Pourquoi un écart minuscule peut-il produire un saut de score ?

In [ ]:
# Faire varier un seul capteur sur deux copies du même lot.
capteur = x_train.columns[arbre.tree_.feature[0]]
coupure = arbre.tree_.threshold[0]
voisins = pd.concat([x_val.iloc[[position]]] * 2, ignore_index=True)
voisins[capteur] = [coupure - 0.01, coupure + 0.01]
# Comparez arbre.apply(voisins) puis arbre.predict_proba(voisins)[:, 1].
resultats = voisins[[capteur]].assign(
    feuille=arbre.apply(voisins), score=arbre.predict_proba(voisins)[:, 1]
)
display(resultats.assign(inspection=resultats.score >= 0.5))

**Corrigé.** Pour temperature, les deux observations artificielles aboutissent aux feuilles 2 et 6 et reçoivent des scores de 0.069 et 0.750. Chaque feuille renvoie une fréquence constante. Franchir la coupure remplace le groupe de référence par un autre. Avant une décision sensible près du seuil, Camille peut vérifier la précision du capteur.

### Jusqu’où laisser pousser l’arbre ?

![Effectif et nombre de feuilles selon la profondeur maximale](../figures/tp04-profondeur.svg)

La **profondeur** est le nombre de questions successives depuis la racine. Une profondeur maximale plus grande autorise des groupes plus fins. L’effectif minimal par feuille impose ici de conserver au moins douze lots anciens par groupe. Un arbre qui épouse des particularités du train peut perdre en qualité sur les lots suivants : c’est le **surapprentissage**. On peut le rechercher en comparant les erreurs sur les deux périodes au même seuil.

### À vous | Comparer deux profondeurs

Ajustez un arbre de profondeur maximale 6. Pour les arbres de profondeur 2 et 6, affichez le nombre de feuilles et la proportion d’erreurs sur le train puis sur la validation, avec le même seuil inclusif de 0,50. Une baisse d’erreur sur les lots anciens suffit-elle pour le préférer ?

In [ ]:
# Isoler l’effet de la profondeur en conservant les autres paramètres.
profond = ajuster_arbre(6)
# Pour chaque modèle, comparez predict_proba(x)[:, 1] >= 0.5 au verdict réel.
for m in [arbre, profond]:
    print(
        m.get_n_leaves(),
        [
            round(np.mean((m.predict_proba(x)[:, 1] >= 0.5) != lots.defaut), 3)
            for x, lots in [(x_train, train), (x_val, val)]
        ],
    )

**Corrigé.** L’arbre court puis l’arbre profond donnent respectivement 4 feuilles (16.7% d’erreurs sur train, 24.2% en validation) et 29 feuilles (11.2% d’erreurs sur train, 22.5% en validation). Ces proportions donnent le même poids à chaque erreur. Avant de recommander un modèle, il faut examiner les coûts asymétriques et les places disponibles.

## 3. Combiner plusieurs arbres

**Atelier 4.2 du support MLB.** Sections 3 à 4.

![Deux organisations des arbres : entraînements indépendants ou corrections successives](../figures/tp04-ensembles.svg)

**Bagging et forêt aléatoire.** Pour chaque arbre, on tire avec remise autant de lignes que dans le train : certaines reviennent plusieurs fois et d’autres sont absentes. C’est un échantillon *bootstrap*. Le bagging agrège des modèles entraînés sur ces tirages. Une forêt aléatoire diversifie aussi les questions possibles en tirant un sous-ensemble de variables à chaque nœud. Dans scikit-learn, son score binaire est la **moyenne des scores de défaut** des arbres. L’intérêt est de réduire la sensibilité à un seul échantillon, surtout si les arbres ne commettent pas toujours les mêmes erreurs.

![Exemple de tirage avec remise et lots absents](../figures/tp04-bootstrap.svg)

Dans ce tirage miniature, six places ne représentent que quatre lots distincts. Deux lots sont absents et deux sont tirés deux fois.

**XGBoost et gradient boosting.** XGBoost utilise le gradient et la courbure de la perte pour choisir les coupures et la valeur des feuilles. La régularisation limite les corrections trop fortes. Les petits arbres sont ajustés l’un après l’autre. Chaque nouvel arbre apprend à corriger le score courant. En classification binaire avec perte logistique, la direction de correction dépend de la différence entre verdict et probabilité courante. Le taux d’apprentissage réduit l’amplitude de chaque ajout. Les contributions s’additionnent sur une échelle de score, puis une fonction logistique convertit le total en probabilité. Moyenner les probabilités des étapes donnerait un autre calcul.

Pour lire les sorties, prenez trois arbres fictifs qui attribuent au même lot les scores 0,2, 0,4 et 0,9. Leur moyenne vaut 0,5. Avec un seuil inclusif de 0,5, la forêt signale le lot, même si un seul arbre dépasse ce seuil individuellement. Dans XGBoost, chaque nouvel arbre ajuste le score construit jusque-là. Faire la moyenne de trois probabilités décrirait donc un autre calcul.

In [ ]:
# Comparer deux ensembles en les entraînant sur les mêmes lots et les mêmes entrées.
foret = RandomForestClassifier(
    n_estimators=80,
    min_samples_leaf=5,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
).fit(x_train, train.defaut)
boost = XGBClassifier(
    n_estimators=80,
    learning_rate=0.06,
    max_depth=2,
    min_child_weight=1,
    tree_method="hist",
    n_jobs=1,
    random_state=42,
).fit(x_train, train.defaut)
modeles = {
    "Arbre court": arbre,
    "Arbre profond": profond,
    "Forêt": foret,
    "XGBoost": boost,
}
display(
    pd.DataFrame(
        {
            nom: m.predict_proba(x_val.iloc[[position]])[:, 1]
            for nom, m in modeles.items()
        },
        index=[val.iloc[position].id],
    ).round(3)
)

### À vous | Entendre les voix de la forêt

Pour le lot étudié, calculez le score donné par chacun des 80 arbres. Comparez trois nombres : la moyenne de ces scores, la proportion d’arbres dont le score atteint 0,50 et le score de la forêt. Quel calcul reproduit exactement la sortie de scikit-learn ? Pourquoi la distinction est-elle importante près du seuil d’inspection ?

In [ ]:
# Recueillir les avis sur un seul panneau avant toute agrégation.
panneau = x_val.iloc[[position]].to_numpy()
avis = np.array([m.predict_proba(panneau)[0, 1] for m in foret.estimators_])
print(avis[:8].round(3))
# Comparez avis.mean() et (avis >= 0.5).mean() au score de la forêt.
score_foret = foret.predict_proba(x_val.iloc[[position]])[0, 1]
print(avis.mean(), (avis >= 0.5).mean(), score_foret)
assert np.isclose(avis.mean(), score_foret)

**Corrigé.** Le score moyen vaut 0.760 et la proportion de votes positifs vaut 0.775. Le premier reproduit le score de la forêt. Transformer chaque avis en oui/non ferait perdre la différence entre un arbre hésitant à 0,51 et un arbre à 0,99. Près du seuil, cette perte peut changer la priorité.

### Diagnostic | Les arbres de la forêt sont-ils d’accord ?

![Répartition des 80 scores individuels pour le même panneau](../figures/tp04-dispersion.svg)

Une moyenne ne dit pas comment les arbres la produisent. Sur la figure, chaque barre compte les arbres dans une tranche de scores. Le trait vertical indique leur moyenne. Comparez les effectifs aux deux extrémités. Calculez ensuite la médiane et l’intervalle entre les quartiles (les 25e et 75e percentiles). Pourquoi cet intervalle ne donne-t-il pas une garantie sur le verdict du lot ?

In [ ]:
# Décrire la dispersion des scores que la moyenne de la forêt résume.
bornes_scores = np.linspace(0, 1, 6)
# np.histogram(avis, bins=bornes_scores) compte les arbres par tranche.
# np.quantile(avis, [0.25, 0.50, 0.75]) donne les trois quartiles.
effectifs, _ = np.histogram(avis, bins=bornes_scores)
quartiles = np.quantile(avis, [0.25, 0.50, 0.75])
print("Arbres par tranche :", effectifs, "quartiles :", quartiles.round(3))
assert effectifs.sum() == foret.n_estimators

**Corrigé.** Pour ce panneau, 6 arbres sur 80 donnent un score inférieur à 0,20 et 50 un score supérieur ou égal à 0,80. La moitié centrale des scores se situe entre 0.598 et 1.000. Ces arbres partagent de nombreuses observations d’entraînement. Leur dispersion décrit le désaccord interne de la forêt ; elle ne constitue pas un intervalle de confiance pour le risque réel.

### Calcul | Que cherche à corriger le prochain arbre de boosting ?

![Correction demandée pour quatre combinaisons de probabilité et de verdict](../figures/tp04-residus.svg)

À une étape du boosting logistique, on dispose d’une probabilité actuelle $p$ et d’un verdict historique $y$. Le **pseudo-résidu** est le négatif du gradient de la perte logistique par rapport au score brut. Pour un verdict binaire, il s’écrit simplement

$$
r=y-p.
$$

Un résidu positif demande d’augmenter le score, un résidu négatif de le diminuer. Le prochain arbre rapproche les lots dont les corrections se ressemblent. La valeur finale de chaque feuille est ajustée pour la perte logistique, puis multipliée par le taux d’apprentissage.

Pour les quatre cas fictifs de la figure, classez les corrections par amplitude. Pour chacun, indiquez s’il faut monter ou descendre. Que se passerait-il si on ajoutait directement chaque résidu à la probabilité du même lot ?

In [ ]:
# Séparer la direction et l’amplitude de la correction sur quatre lots historiques.
mini = pd.DataFrame(
    {"lot": ["A", "B", "C", "D"], "p": [0.1, 0.8, 0.2, 0.9], "y": [1, 1, 0, 0]}
)
# Ajoutez une colonne residu, puis triez selon sa valeur absolue.
mini["residu"] = mini.y - mini.p
display(mini.sort_values("residu", key=lambda s: s.abs(), ascending=False))
assert np.allclose(mini.p + mini.residu, mini.y)

**Corrigé.** A et D nécessitent les corrections les plus fortes (+0,9 et −0,9). B et C ont déjà une probabilité plus proche de leur verdict (+0,2 et −0,2). Ajouter y − p à p restituerait exactement y sur chaque exemple observé. Cela supposerait de connaître le verdict du prochain panneau. Le boosting apprend une règle de correction partagée à partir des capteurs et l’applique sur l’échelle de score brut.

### Du résidu à la valeur d’une feuille

![Deux défauts et un conforme dans la même feuille fictive](../figures/tp04-feuille-poids.svg)

Ces trois lots tombent dans la même feuille. Quelle correction commune proposer alors que le lot C réclame une baisse ? Pour la perte logistique, chaque lot contribue un gradient $g=p-y$ et une courbure $h=p(1-p)$. Pour une feuille de structure fixée, avec une pénalisation L2 de force $\lambda$ et sans pénalisation L1, XGBoost calcule le poids

$$
w=-\frac{\sum_i g_i}{\sum_i h_i+\lambda}.
$$

Le taux d’apprentissage multiplie ensuite ce poids. Pour cet exemple, utilisez un taux de 0,30 et un score brut initial nul (probabilité 0,50). Calculez la probabilité après un arbre pour λ = 0 puis λ = 1. Expliquez pourquoi la valeur de la feuille agit aussi sur le lot conforme.

![Effet de la pénalité L2 à taux et feuille constants](../figures/tp04-regularisation.svg)

À mesure que λ augmente, vers quel score la courbe revient-elle ? Le tracé maintient les lots et la structure de la feuille constants.

In [ ]:
# Calculer une correction partagée pour trois panneaux dans une feuille imposée.
verdicts_feuille = np.array([1, 1, 0])
probas_feuille = np.full(3, 0.5)
gradient = probas_feuille - verdicts_feuille
courbure = probas_feuille * (1 - probas_feuille)
# Additionnez les contributions avant de calculer le poids de la feuille.
for penalite in (0, 1):
    poids = -gradient.sum() / (courbure.sum() + penalite)
    score = 1 / (1 + np.exp(-0.30 * poids))
    print(penalite, round(poids, 3), round(score, 3))

**Corrigé.** Le gradient total vaut −0,5 et la courbure totale 0,75. Les poids valent 0,667 puis 0,286. Les scores montent à 0,550 et 0,521. Les trois lots suivent la même branche et reçoivent le même incrément de score brut. La pénalité réduit le poids appris, tandis que le taux ralentit son application.

## 4. Retenir deux candidats pour la suite

![Coût et inspections des règles candidates sur la validation](../figures/tp04-selection.svg)

Chaque point correspond à un modèle et un seuil. Pour comparer les quatre modèles, on essaie la même grille de seuils de 0,05 à 1 par pas de 0,05. La règle est inclusive. Dans cet exercice, on dispose de **48 places pour 240 lots**. Parmi les règles réalisables, on minimise le coût ci-dessous, puis la charge. À égalité pour un même modèle, on retient le seuil le plus élevé.

$$
C = 1 \times FP + 10 \times FN.
$$

Un faux positif (FP) est un lot conforme inspecté. Un faux négatif (FN) est un défaut manqué. Un défaut intercepté consomme aussi une place. Comme au TP 02, la référence sans aucune inspection manque les 78 défauts et coûte 780 points. Le test final demeure réservé au TP 08.

In [ ]:
# Évaluer les mêmes seuils sur les mêmes lots sans consulter le test final.
def grille(modele, nom):
    s = modele.predict_proba(x_val)[:, 1]
    y = val.defaut.to_numpy() == 1
    lignes = []
    for centieme in range(5, 101, 5):
        seuil = centieme / 100
        alerte = s >= seuil
        fp, fn = int((alerte & ~y).sum()), int((~alerte & y).sum())
        lignes.append(
            {
                "modèle": nom,
                "seuil": seuil,
                "inspections": int(alerte.sum()),
                "VP": int((alerte & y).sum()),
                "FP": fp,
                "FN": fn,
                "coût": fp + 10 * fn,
            }
        )
    return pd.DataFrame(lignes)


essais = pd.concat([grille(m, nom) for nom, m in modeles.items()], ignore_index=True)


def meilleurs(tableau, places):
    possibles = tableau.loc[tableau.inspections <= places]
    return possibles.sort_values(
        ["coût", "inspections", "seuil", "modèle"], ascending=[True, True, False, True]
    ).drop_duplicates("modèle")


display(essais.loc[essais.seuil == 0.5])

### Préparer la note | Retenir deux règles distinctes

Écartez les règles irréalisables avant de trier. Retenez deux **modèles distincts**. Indiquez pour chacun son seuil, sa charge, les défauts interceptés et manqués, le coût et le gain sur la référence. Reprenez le lot du début : un candidat l’inspecterait-il au seuil retenu ? Distinguez cette décision du verdict final. Quelle limite de cette comparaison faut-il rappeler à l’atelier ?

In [ ]:
# Sélectionner une seule règle admissible par modèle avant de comparer les candidats.
classement = meilleurs(essais, 48)
# Consultez classement.head(2), puis appliquez les deux seuils au panneau étudié.
finalistes = classement.head(2)
display(finalistes.assign(gain=780 - finalistes["coût"]))
for _, r in finalistes.iterrows():
    s = modeles[r["modèle"]].predict_proba(x_val.iloc[[position]])[0, 1]
    print(r["modèle"], round(s, 3), "inspection :", s >= r["seuil"])
assert finalistes["modèle"].nunique() == 2 and finalistes.inspections.max() <= 48

**Corrigé.** XGBoost à 0.60 demande 47 contrôles, intercepte 42 défauts et en manque 36. Le coût atteint 365 points, soit 415 de moins que la référence. Forêt à 0.60 demande 47 contrôles, intercepte 40 défauts et en manque 38. Le coût atteint 387 points, soit 393 de moins que la référence. La validation a servi au réglage et à la sélection. Ces gains ne constituent pas une estimation indépendante du résultat futur. Le chemin d’un seul arbre explique exactement sa propre décision, mais ne résume pas les autres arbres d’un ensemble.

### Arbitrage | Les deux finalistes inspectent-ils les mêmes lots ?

![Nombre de conformes et de défauts pour chaque combinaison de décisions](../figures/tp04-recoupement.svg)

Les deux candidats respectent le budget. Inspecteraient-ils les mêmes panneaux ? Formez quatre groupes à partir des deux décisions. Dans chaque groupe, comptez les lots et les défauts. Évaluez la suggestion du chef de presse : « Inspectons tout lot retenu par au moins un des deux modèles. » Comptez la charge et les défauts supplémentaires par rapport au premier candidat.

Si Camille conserve le premier candidat comme seule règle, combien de fausses alertes et de défauts manqués l’union changerait-elle ? Recalculez la variation du coût, puis décidez si ce gain serait réalisable avec 48 places.

In [ ]:
# Apparier les décisions par lot en appliquant le seuil propre à chaque finaliste.
duo = classement.head(2)
decisions = {}
for _, regle in duo.iterrows():
    nom = regle["modèle"]
    decisions[nom] = modeles[nom].predict_proba(x_val)[:, 1] >= regle["seuil"]
premier, second = duo["modèle"]
audit_duo = pd.DataFrame(decisions).assign(defaut=val.defaut.to_numpy())
# Groupez par les deux colonnes de décision. Le « ou » entre deux masques se note |.
groupes_duo = audit_duo.groupby([premier, second]).defaut.agg(["size", "sum"])
display(groupes_duo)
union = audit_duo[premier] | audit_duo[second]
ajout = union & ~audit_duo[premier]
print(
    "Charge combinée :",
    union.sum(),
    "défauts supplémentaires :",
    audit_duo.loc[ajout, "defaut"].sum(),
)

**Corrigé.** Les 44 lots retenus en commun comprennent 40 défauts. XGBoost seul ajoute trois lots, dont deux défectueux. La forêt seule ajoute trois lots conformes. L’union occupe donc 50 places et manque toujours 36 défauts. Elle coûte 368 points, soit trois de plus que XGBoost seul, et dépasse le budget de deux inspections.

**À remettre à Camille.** Rassemblez le chemin du lot, le tableau des deux candidats et votre réponse à la proposition de cumuler leurs alertes. Précisez le nombre de places et la période utilisée.

## Dernier créneau d’inspection | Quelle ligne choisir ?

Camille peut inspecter un lot sur A ou un lot sur B. Sur A, 12 des 100 derniers lots inspectés avaient un défaut. Sur B, un seul des 5 lots inspectés en avait un. Pour cet exemple, les lots inspectés jusqu’ici sont représentatifs de leur ligne.

| Ligne | Lots défectueux détectés | Lots inspectés |
| --- | ---: | ---: |
| A | 12 | 100 |
| B | 1 | 5 |

Quelle ligne choisiriez-vous pour le prochain contrôle ? Calculez les deux taux. Si le prochain lot de B est sain, que devient son taux ? Expliquez à Camille comment votre décision dépend du nombre de créneaux qu’il reste : un seul aujourd’hui ou un créneau chaque jour.

**Corrigé.** A donne 12/100 = 12 % et B donne 1/5 = 20 %. En suivant uniquement le taux observé, on inspecte B. Si le prochain lot de B est sain, la proportion tombe à 1/6, soit environ 16,7 %. Cinq inspections donnent peu de précision : le classement observé peut encore changer.

S’il reste un seul contrôle, apprendre le taux de B ne rapporte rien à cette série de décisions. Choisir B sur la base des taux est défendable, mais les données ne garantissent pas que B soit réellement la ligne la plus risquée. Avec des contrôles futurs, inspecter B peut aussi renseigner les décisions suivantes. Dans cet exemple, exploiter la meilleure estimation et explorer la ligne la moins connue conduisent au même choix pour deux raisons différentes.

Un **bandit à deux bras** représente ces choix successifs. Le gain vaut 1 pour un défaut détecté et 0 pour un lot sain. À chaque tour, Camille voit le verdict de la ligne inspectée. Pour comparer les stratégies, la simulation suppose le verdict disponible avant le tour suivant ; dans l’atelier, il faut tenir compte du délai du laboratoire.

### Une stratégie optimiste choisit-elle la même presse ?

![Le bonus relie le taux observé de chaque presse à son indice optimiste.](../figures/tp04-bonus-ucb.svg)

Après une autre campagne, A compte 20 défauts sur 100 contrôles et B, 1 sur 10. Une stratégie gloutonne choisit le meilleur taux observé. UCB1 ajoute à chaque taux un bonus qui diminue avec le nombre de contrôles de la ligne. Si n est ce nombre et t le total des contrôles, l’indice vaut

$$
U = \widehat p + \sqrt{2\ln(t)/n}.
$$

Prédisez les deux choix avant d’exécuter. Puis remplacez pour B « 1 sur 10 » par « 10 sur 100 ». Son taux reste identique. Que devient l’avantage donné à l’exploration ?

In [ ]:
# Comparer le prochain choix selon le rendement observé et le bonus d’exploration.
for controles_b, defauts_b in [(10, 1), (100, 10)]:
    controles = np.array([100, controles_b])
    taux = np.array([20, defauts_b]) / controles
    indice = taux + np.sqrt(2 * np.log(controles.sum()) / controles)
    print("Contrôles sur B :", controles_b, "| indices A, B :", indice.round(3))
    print("Gloutonne :", "AB"[taux.argmax()], "| UCB1 :", "AB"[indice.argmax()])

**Corrigé.** Avec dix contrôles sur B, A a le meilleur taux (20 % contre 10 %), mais B reçoit le plus grand bonus. UCB1 explore B. À cent contrôles par ligne, les bonus sont égaux et les deux stratégies choisissent A. L’indice UCB peut dépasser 1 car il sert à classer les choix. Il ne représente pas la probabilité d’un défaut.

## Défi facultatif | Combien de lots différents un arbre de la forêt voit-il ?

Pour 720 tirages, distinguez le nombre de lignes tirées du nombre de lots distincts. Pour un lot donné, la probabilité d’être absent à un tirage vaut 719/720. Élevez-la à la puissance 720 pour estimer la proportion moyenne de lots absents de tout le tirage. Comparez avec les indices réellement tirés pour les 80 arbres. Pourquoi ces absences ne remplacent-elles pas une validation sur des dates ultérieures ?

In [ ]:
# Compter les identifiants distincts parmi les indices effectivement tirés.
tirages = foret.estimators_samples_
n = len(train)
# np.unique(indices).size compte les lots distincts dans un tirage.
absents = np.array([n - np.unique(indices).size for indices in tirages])
print("Absence attendue :", (1 - 1 / n) ** n, "observée :", absents.mean() / n)
print("Premier arbre :", len(tirages[0]), "tirages,", n - absents[0], "lots distincts")

**Corrigé.** Pour le premier arbre, il y a 720 tirages mais 456 lots distincts. En moyenne, 36.9% des lots manquent par arbre, près des 36,8 % attendus. Un lot omis d’un tirage appartient toujours à la période ancienne. Cela ne simule pas un changement du procédé dans le temps.

## Défi facultatif | Observer le boosting en cours d’entraînement

![Score d’un même lot après chaque ajout d’arbre](../figures/tp04-etapes.svg)

Pour le lot du début, notez le score après 1, 10, 40 et 80 étapes. Vérifiez le dernier avec `predict_proba`. Le score monte-t-il nécessairement à chaque ajout ? Calculez ensuite le coût au seuil fixe de 0,50 sur la validation à ces quatre étapes. Pourquoi ne peut-on pas choisir le nombre d’arbres en observant un seul lot ?

In [ ]:
# Observer les prédictions cumulées après chaque ajout, sans réentraîner le modèle.
etapes = [1, 10, 40, 80]
# iteration_range=(0, k) utilise les k premiers arbres déjà appris.
trajectoire = []
for k in range(1, 81):
    proba = boost.predict_proba(x_val, iteration_range=(0, k))
    trajectoire.append(proba[position, 1])
    if k in etapes:
        alerte = proba[:, 1] >= 0.5
        fp = (alerte & val.defaut.eq(0)).sum()
        fn = (~alerte & val.defaut.eq(1)).sum()
        print(k, round(proba[position, 1], 3), int(fp + 10 * fn))
assert np.isclose(trajectoire[-1], boost.predict_proba(x_val.iloc[[position]])[0, 1])

**Corrigé.** Le score du lot passe de 0,290 à 0,712. Chaque ajout vise la perte sur l’ensemble du train ; il peut relever ou abaisser le score individuel. Les coûts affichés évaluent toute la validation mais à un seuil fixe. Pour une présélection opérationnelle, il faut aussi contrôler la charge.

## Défi facultatif | La liste résiste-t-elle à un changement de capacité ?

Reprenez la figure coût-charge de la section 4. Prévoyez quels points deviennent accessibles si la limite passe de 48 à 60, puis recommencez avec 36. Retrouvez les deux modèles retenus pour chaque budget. Un modèle peut-il changer de seuil sans changer de rang ? Quel risque y aurait-il à promettre que le volume des alertes restera exactement le même sur les prochains lots ?

In [ ]:
# Refaire uniquement la sélection pour isoler l’effet d’un changement de capacité.
budgets = [36, 48, 60]
# Appelez meilleurs(essais, places) et affichez les deux premières lignes.
for places in budgets:
    print("Places disponibles :", places)
    display(meilleurs(essais, places).head(2))

**Corrigé.** Les premières places sont occupées par Forêt au seuil 0.75 pour 36 places, XGBoost au seuil 0.60 pour 48 places, XGBoost au seuil 0.50 pour 60 places. Quand le budget augmente, les anciennes règles restent réalisables : le meilleur coût possible ne peut donc pas augmenter sur cette même grille. Sur de nouveaux lots, la distribution des scores peut changer et avec elle le nombre d’alertes.

## Défi facultatif | De quel capteur la forêt dépend-elle ?

![Perte de rappel moyen après permutation de chaque entrée, à seuil fixe](../figures/tp04-permutation.svg)

Une **importance par permutation** mesure la dégradation d’un indicateur quand on mélange une entrée entre les lots. On conserve les autres entrées, le modèle et le seuil. Ici l’indicateur est le rappel (part des défauts interceptés). Une valeur positive indique une baisse du rappel. Sur la figure, les segments couvrent les résultats de cinq mélanges par variable.

Reproduisez le calcul pour les quatre colonnes de validation. Pour un mélange, vérifiez que les valeurs sont conservées et que le nombre d’alertes peut changer. Quel capteur arrive en tête ? Pourquoi cette enquête ne permet-elle pas de recommander de modifier un réglage physique ?

In [ ]:
# Perturber une entrée à la fois en gardant le seuil sélectionné pour la forêt.
seuil_foret = float(classement.loc[classement["modèle"].eq("Forêt"), "seuil"].iloc[0])


def bilan_foret(x):
    alerte = foret.predict_proba(x)[:, 1] >= seuil_foret
    return alerte[val.defaut.to_numpy() == 1].mean(), alerte.sum()


rappel_base, charge_base = bilan_foret(x_val)
# Utilisez une copie de x_val et rng.permutation pour conserver toutes les valeurs.
# Répétez pour chaque colonne et comparez le rappel à rappel_base.
rng = np.random.default_rng(42)
pertes = {}
for colonne in x_val:
    pertes[colonne] = []
    for repetition in range(5):
        melange = x_val.copy()
        melange[colonne] = rng.permutation(x_val[colonne])
        rappel, charge = bilan_foret(melange)
        pertes[colonne].append(100 * (rappel_base - rappel))
        if repetition == 0:
            assert np.array_equal(np.sort(melange[colonne]), np.sort(x_val[colonne]))
            print(colonne, "charge après mélange :", charge, "au lieu de", charge_base)
display(pd.DataFrame(pertes).agg(["mean", "min", "max"]).round(1))

**Corrigé.** Le mélange de temperature produit la plus forte perte moyenne de rappel (20.3 points sur cinq mélanges). Cette quantité dépend du modèle, du seuil, de la période et de l’indicateur. Le mélange peut aussi créer des combinaisons de mesures rares en usine. On a testé la dépendance de la prédiction aux informations d’un capteur, pas l’effet causal d’un changement de procédé.

### Pour approfondir

[Arbres de décision dans scikit-learn](https://scikit-learn.org/stable/modules/tree.html) | [Forêts, bagging et gradient boosting](https://scikit-learn.org/stable/modules/ensemble.html) | [Fonction objectif et arbres dans XGBoost](https://github.com/dmlc/xgboost/blob/master/doc/tutorials/model.rst)